# Feature Inspection

This notebook lets you:
1. Run feature extraction and see the output
2. Load an existing `features.csv` and concat new patients onto it
3. Check NaN counts per subject and per feature
4. See exactly which patients are incomplete and what they're missing

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import gzip, shutil
import os

## Settings
Edit these paths to match your setup.

In [ ]:
MERGED_DIR     = Path("merged_csv_eog_backup")      # !!! Change this to desired path or set to None to skip patient info
FEATURE_CSV    = Path("features_csv/features.csv")  # !!! Change this to desired path or set to None to skip patient info
PATIENT_EXCEL  = Path("path/to/patient_info.xlsx")  # !!! Change this to desired path or set to None to skip patient info
FS             = 250.0
PATTERN        = "*_merged.csv"

---
## Compress merged CSVs

Run this to compress all merged CSVs in `merged_csv_eog` and delete the uncompressed versions. This is optional but saves a lot of disk space. You can still read the compressed CSVs with `pd.read_csv(..., compression='gzip')`

In [ ]:
# compress_merged.py  (delete after running)

for f in Path("merged_csv_eog").glob("*_merged.csv"):
    gz = f.with_suffix(".csv.gz")
    with open(f, "rb") as fi, gzip.open(gz, "wb", 6) as fo:
        shutil.copyfileobj(fi, fo)
    saved = f.stat().st_size - gz.stat().st_size
    print(f"{f.name}: freed {saved / 1024**2:.1f} MB")
    f.unlink()

---
## Size check
This cell checks the size of the merged CSVs before and after compression, and prints out how much space was saved for each file and in total. You can run this after the compression step to verify that it worked as expected.

In [ ]:
rows = []
for f in sorted(MERGED_DIR.glob("*.csv.gz")):
    size_mb = f.stat().st_size / 1e6
    try:
        df = pd.read_csv(f, usecols=["time_sec"], low_memory=True)
        duration_min = df["time_sec"].iloc[-1] / 60
        n_samples = len(df)
    except Exception as e:
        duration_min = None
        n_samples = None
    rows.append({
        "file": f.name,
        "size_mb": round(size_mb, 1),
        "n_samples": n_samples,
        "duration_min": round(duration_min, 1) if duration_min else None,
    })

df_sizes = pd.DataFrame(rows).sort_values("size_mb", ascending=False)
print(df_sizes.to_string(index=False))

---
## 1. Full extraction (from scratch)
Run this to extract features for **all** merged CSVs. This overwrites `features.csv`.

In [ ]:
from analysis.feat_report import collect_features

combined = collect_features(
    merged_dir=MERGED_DIR,
    fs=FS,
    pattern=PATTERN,
    patient_excel=PATIENT_EXCEL,
)

print(f"\nShape: {combined.shape}")
combined.head()

In [ ]:
# Save
FEATURE_CSV.parent.mkdir(parents=True, exist_ok=True)
combined.to_csv(FEATURE_CSV, index=False)
print(f"Saved: {FEATURE_CSV}  ({combined.shape[0]} subjects, {combined.shape[1] - 1} features)")

---
## 2. Incremental extraction (concat new patients)
Only extract features for patients not already in `features.csv`, then concat.

In [ ]:
from analysis.feat_report import collect_features

# Load existing
existing_df = pd.read_csv(FEATURE_CSV)
existing_subjects = set(existing_df["subject_id"].tolist())
print(f"Existing features: {len(existing_subjects)} subjects")

# Find new merged CSVs
all_files = sorted(MERGED_DIR.glob(PATTERN))
new_files = [f for f in all_files if f.stem not in existing_subjects]
print(f"Total merged CSVs: {len(all_files)}")
print(f"New (not in CSV):  {len(new_files)}")

if new_files:
    print(f"\nNew files:")
    for f in new_files:
        print(f"  {f.name}")
else:
    print("\nNo new files to extract.")

In [ ]:
# Extract only the new ones
if new_files:
    new_df = collect_features(
        merged_dir=MERGED_DIR,
        fs=FS,
        pattern=PATTERN,
        patient_excel=PATIENT_EXCEL,
        file_list=new_files,
    )
    print(f"\nNew features: {new_df.shape}")
    
    # Check column mismatch
    old_cols = set(existing_df.columns)
    new_cols = set(new_df.columns)
    
    only_in_new = new_cols - old_cols
    only_in_old = old_cols - new_cols
    
    if only_in_new:
        print(f"\nWARNING: New columns not in existing data: {sorted(only_in_new)}")
        print(f"Existing subjects will have NaN for these columns.")
    if only_in_old:
        print(f"\nWARNING: Existing columns missing from new data: {sorted(only_in_old)}")
    
    # Concat
    combined = pd.concat([existing_df, new_df], ignore_index=True)
    print(f"\nCombined: {combined.shape[0]} subjects, {combined.shape[1]} columns")
else:
    combined = existing_df
    print("Nothing new — using existing data.")

In [ ]:
# Save the combined result
# Uncomment the line below to overwrite features.csv

# combined.to_csv(FEATURE_CSV, index=False)
# print(f"Saved: {FEATURE_CSV}")

---
## 3. NaN inspection
Load `features.csv` and check for missing values.

In [ ]:
df = pd.read_csv(FEATURE_CSV)
print(f"Loaded: {df.shape[0]} subjects, {df.shape[1]} columns")

### 3a. NaN count per feature
Which features have the most missing values?

In [ ]:
nan_per_feature = df.isna().sum()
nan_per_feature = nan_per_feature[nan_per_feature > 0].sort_values(ascending=False)

if nan_per_feature.empty:
    print("No NaN values in any feature!")
else:
    print(f"{len(nan_per_feature)} features have NaN values:\n")
    for feat, count in nan_per_feature.items():
        pct = count / len(df) * 100
        print(f"  {feat:<45s}  {count:3d} / {len(df)}  ({pct:.1f}%)")

### 3b. NaN count per subject
Which subjects have missing values, and how many?

In [ ]:
id_cols = ["subject_id", "DCSM_ID"]
feature_cols = [c for c in df.columns if c not in id_cols and pd.api.types.is_numeric_dtype(df[c])]

nan_per_subject = df[feature_cols].isna().sum(axis=1)
subjects_with_nan = df[nan_per_subject > 0][["subject_id"]].copy()
subjects_with_nan["nan_count"] = nan_per_subject[nan_per_subject > 0].values

if subjects_with_nan.empty:
    print("All subjects have complete features!")
else:
    print(f"{len(subjects_with_nan)} subject(s) have NaN values:\n")
    for _, row in subjects_with_nan.sort_values("nan_count", ascending=False).iterrows():
        print(f"  {row['subject_id']:<50s}  {row['nan_count']:.0f} missing features")

### 3c. Detailed NaN report
For each subject with NaN values, show exactly which features are missing.

In [ ]:
rows_with_nan = df[df[feature_cols].isna().any(axis=1)]

if rows_with_nan.empty:
    print("No NaN values found.")
else:
    for _, row in rows_with_nan.iterrows():
        sid = row["subject_id"]
        missing = [c for c in feature_cols if pd.isna(row[c])]
        print(f"\n{sid}  ({len(missing)} missing):")
        for col in missing:
            print(f"    - {col}")

### 3d. Group label check
Check which subjects have group labels (Control, iRBD, etc.) and which don't.

In [ ]:
group_cols = [c for c in ["Control", "PD(-RBD)", "PD(+RBD)", "iRBD", "PLM"] if c in df.columns]

if not group_cols:
    print("No group columns found. Run extract with --patient-excel to add them.")
else:
    has_label = df[group_cols].sum(axis=1) > 0
    n_labelled = has_label.sum()
    n_unlabelled = (~has_label).sum()
    
    print(f"With group label:     {n_labelled}")
    print(f"Without group label:  {n_unlabelled}")
    
    if n_unlabelled > 0:
        unlabelled = df.loc[~has_label, "subject_id"].tolist()
        print(f"\nUnlabelled subjects:")
        for sid in unlabelled:
            print(f"  - {sid}")
    
    # Group distribution
    print(f"\nGroup distribution:")
    for col in group_cols:
        n = (df[col] == 1).sum()
        print(f"  {col:<12s}  n = {n}")

---
## 4. Recording duration check

Check the recording duration for each subject to identify any that are unusually short, which could explain missing features.

In [ ]:
duration_check = df[["subject_id", "total_duration_min", "rem_duration_min"]].copy()
duration_check["rem_fraction"] = duration_check["rem_duration_min"] / duration_check["total_duration_min"]

print("=== Suspiciously long recordings (>600 min) ===")
print(duration_check[duration_check["total_duration_min"] > 600].to_string(index=False))

print("\n=== Suspiciously short REM (<5 min) ===")
print(duration_check[duration_check["rem_duration_min"] < 5].to_string(index=False))

print("\n=== No REM at all ===")
print(duration_check[duration_check["rem_duration_min"] == 0].to_string(index=False))

---
## 5. Quick summary

In [ ]:
print(f"Total subjects:       {len(df)}")
print(f"Total features:       {len(feature_cols)}")
print(f"Subjects with NaN:    {len(rows_with_nan)}")
print(f"Features with NaN:    {len(nan_per_feature)}")
print(f"Total NaN values:     {df[feature_cols].isna().sum().sum()}")